In [ ]:
import pandas as pd

In [15]:
obj = [
  {
    "amount": 500,
    "category": "Rent",
    "date": "2025-12-20",
    "user_id": 1,
    "id": 1,
    "transaction_type": "income",
    "description": "Ventura"
  },
  {
    "amount": 400,
    "category": "Rent",
    "date": "2025-12-20",
    "user_id": 1,
    "id": 2,
    "transaction_type": "expense",
    "description": "Ventura"
  },
  {
    "amount": 400,
    "category": "Rent",
    "date": "2025-11-20",
    "user_id": 1,
    "id": 3,
    "transaction_type": "expense",
    "description": "Ventura"
  }
]

data = pd.DataFrame(obj)#.set_index(["id"])

data

,amount,category,date,user_id,id,transaction_type,description
0,500,Rent,2025-12-20,1,1,income,Ventura
1,400,Rent,2025-12-20,1,2,expense,Ventura
2,400,Rent,2025-11-20,1,3,expense,Ventura


In [13]:
# make sure date is datetime
df = data.copy()
df["date"] = pd.to_datetime(df["date"])

# month key + display label
df["month"] = df["date"].dt.to_period("M")          # e.g., 2025-12
df["month_label"] = df["date"].dt.strftime("%b-%Y") # e.g., Dec-2025

# aggregate per month
out = (
    df.groupby("month", as_index=False)
      .agg(
          transaction_month=("month_label", "first"),
          total_income=("amount", lambda s: s[df.loc[s.index, "transaction_type"].eq("income")].sum()),
          total_expense=("amount", lambda s: s[df.loc[s.index, "transaction_type"].eq("expense")].sum()),
          most_common_expense_category=("category", lambda s: (
              df.loc[s.index]
                .query("transaction_type == 'expense'")["category"]
                .mode()
                .iat[0] if (df.loc[s.index, "transaction_type"].eq("expense").any()) else None
          )),
      )
)

# optional: sort newest->oldest and use month label as index
out = out.sort_values("month", ascending=False).drop(columns=["month"]).set_index("transaction_month")

out


,total_income,total_expense,most_common_expense_category
transaction_month,,,
Dec-2025,500,400,Rent
Nov-2025,0,400,Rent
Oct-2025,0,400,Rent
